# BD-KDD Kidney Disease — mRMR + DODA Feature Selection

## Experimental objective

Evaluate conventional **mRMR feature selection** against **mRMR + DODA clinical-priority re-ranking** on the BD-KDD kidney-disease dataset.

### Important design rule

mRMR ranks **all 24 predictors**. DODA also receives **all 24 predictor scores** and performs the final Top-K selection after clinical-priority fusion.

This is essential: if mRMR selected Top-K before DODA, DODA could not re-prioritize features outside that preliminary Top-K.

### Dataset

- 988 records
- 24 predictors
- Binary target: `Class`
- `Class = 0`: healthy
- `Class = 1`: kidney disease
- 0 missing values
- 80/20 stratified fixed split for the initial experiment
- 5×5 repeated stratified CV for robustness


In [3]:
# =============================================================================
# STEP 1: LOAD PROCESSED DATASET
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/processed/BD-KDD_cleaned.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("PROCESSED BD-KDD DATASET")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


PROCESSED BD-KDD DATASET
Rows    : 988
Columns : 25


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane,Class
0,58,147,1.025,1,3,0,1,0,0,177,...,43,12342,4.3,1,0,0,1,1,0,0
1,71,142,1.025,2,3,1,0,1,0,247,...,47,7249,3.7,0,1,1,0,1,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,37,4111,3.7,0,0,0,0,0,1,0
3,34,70,1.005,1,3,1,1,0,0,215,...,44,4682,4.0,1,0,1,1,0,1,1
4,62,120,1.020,4,0,1,0,1,0,143,...,54,5161,4.7,0,0,1,0,1,0,1


In [4]:
# =============================================================================
# STEP 2: FEATURE AND TARGET SEPARATION
# =============================================================================

TARGET_COLUMN = "Class"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)
print(f"Predictors (X) : {X.shape[1]}")
print(f"Records        : {X.shape[0]}")
print(f"Target (y)     : {TARGET_COLUMN}")
print(f"Target values  : {sorted(y.unique().tolist())}")

assert X.shape[1] == 24, "Expected 24 predictors after removing Class."
assert set(y.unique()) == {0, 1}

display(X.head())
display(y.head())


FEATURE / TARGET SEPARATION
Predictors (X) : 24
Records        : 988
Target (y)     : Class
Target values  : [0, 1]


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Hemo,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane
0,58,147,1.025,1,3,0,1,0,0,177,...,9.3,43,12342,4.3,1,0,0,1,1,0
1,71,142,1.025,2,3,1,0,1,0,247,...,16.9,47,7249,3.7,0,1,1,0,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,16.6,37,4111,3.7,0,0,0,0,0,1
3,34,70,1.005,1,3,1,1,0,0,215,...,15.4,44,4682,4.0,1,0,1,1,0,1
4,62,120,1.020,4,0,1,0,1,0,143,...,13.9,54,5161,4.7,0,0,1,0,1,0


0    0
1    1
2    0
3    1
4    1
Name: Class, dtype: int64

In [5]:
# =============================================================================
# STEP 3: TARGET DISTRIBUTION
# =============================================================================

target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(target_distribution)


,Count,Percentage
Class,,
0,481,48.68
1,507,51.32


In [6]:
# =============================================================================
# STEP 4: DATA TYPE AND MISSING-VALUE CHECK
# =============================================================================

print("=" * 70)
print("DATA TYPES")
print("=" * 70)
display(X.dtypes.to_frame("Data Type"))

missing = pd.DataFrame({
    "Missing Count": X.isna().sum(),
    "Missing Percentage (%)": (X.isna().mean() * 100).round(2)
})

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing)

assert X.isna().sum().sum() == 0, "Unexpected missing values detected."

print("No missing values detected. No imputation will be performed.")


DATA TYPES


,Data Type
Age,int64
Bp,int64
Sg,float64
Al,int64
Su,int64
Rbc,int64
Pc,int64
Pcc,int64
Ba,int64
Bgr,int64


MISSING VALUES


,Missing Count,Missing Percentage (%)
Age,0,0.0
Bp,0,0.0
Sg,0,0.0
Al,0,0.0
Su,0,0.0
Rbc,0,0.0
Pc,0,0.0
Pcc,0,0.0
Ba,0,0.0
Bgr,0,0.0


No missing values detected. No imputation will be performed.


## Preprocessing decision

The EDA established that BD-KDD contains **zero missing values**. Therefore, no imputation step is included in this notebook.

Feature selection is fitted only on training data/folds. Model scaling is applied only where needed and is fitted only on training data/folds.


In [7]:
# =============================================================================
# STEP 5: FIXED 80/20 STRATIFIED TRAIN-TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nTraining class distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting class distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))


TRAIN-TEST SPLIT
X_train : (790, 24)
X_test  : (198, 24)
y_train : (790,)
y_test  : (198,)

Training class distribution:


Class
1    51.27
0    48.73
Name: proportion, dtype: float64


Testing class distribution:


Class
1    51.52
0    48.48
Name: proportion, dtype: float64

In [8]:
# =============================================================================
# STEP 6: FEATURE-SCALE CHECK
# =============================================================================
# mRMR is applied to the training data after fold-specific preprocessing.
# Scaling is fitted only on the corresponding training partition.
#
# Logistic Regression is scaled again after feature selection.
# Random Forest and XGBoost use the selected features without downstream scaling.

print("mRMR will be fitted using training data only.")
print("Logistic Regression will be scaled after feature selection.")
print("Random Forest and XGBoost will use the selected features without scaling.")


mRMR will be fitted using training data only.
Logistic Regression will be scaled after feature selection.
Random Forest and XGBoost will use the selected features without scaling.


# 1. mRMR Baseline

mRMR is used as the statistical feature-selection baseline.

**Important:** mRMR is fitted on all 24 predictors first. The complete mRMR ranking is generated before Top-K is applied.


In [9]:
# =============================================================================
# STEP 7: mRMR RANKING FOR ALL FEATURES
# =============================================================================

from mrmr import mrmr_classif
from sklearn.preprocessing import StandardScaler

mrmr_scaler = StandardScaler()

X_train_mrmr_scaled = pd.DataFrame(
    mrmr_scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_mrmr_scaled = pd.DataFrame(
    mrmr_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# mRMR expects clean, aligned pandas objects.
X_mrmr_train = X_train_mrmr_scaled.reset_index(drop=True)
y_mrmr_train = y_train.reset_index(drop=True)

mrmr_ranked_features = mrmr_classif(
    X=X_mrmr_train,
    y=y_mrmr_train,
    K=X_mrmr_train.shape[1]
)

# Convert the complete mRMR ordering into decreasing normalized scores.
# Rank 1 = 1.0; rank 24 = 1/24.
n_features = len(mrmr_ranked_features)

mrmr_scores = pd.DataFrame({
    "Feature": mrmr_ranked_features,
    "mRMR_Score": [
        (n_features - rank) / n_features
        for rank in range(n_features)
    ]
})

mrmr_scores["mRMR_Rank"] = np.arange(
    1,
    len(mrmr_scores) + 1
)

display(mrmr_scores)


100%|██████████| 24/24 [00:07<00:00,  3.29it/s]


,Feature,mRMR_Score,mRMR_Rank
0,Bp,1.000000,1
1,Su,0.958333,2
2,Sg,0.916667,3
3,Ba,0.875000,4
4,Bgr,0.833333,5
5,Ane,0.791667,6
6,Bu,0.750000,7
7,Pcv,0.708333,8
8,Sc,0.666667,9
9,Rbcc,0.625000,10


In [10]:
# =============================================================================
# STEP 8: TOP-K mRMR FEATURE SETS
# =============================================================================

TOP_K_VALUES = [5, 10, 15, 20]

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")

mrmr_results = {}

for top_k in TOP_K_VALUES:

    selected = (
        mrmr_scores
        .head(top_k)["Feature"]
        .tolist()
    )

    mrmr_results[top_k] = {
        "features": selected,
        "X_train": X_train[selected].copy(),
        "X_test": X_test[selected].copy()
    }

    print("=" * 70)
    print(f"mRMR TOP-{top_k}")
    print("=" * 70)
    print(selected)


Total predictors: 24
Top-K values: [5, 10, 15, 20]
mRMR TOP-5
['Bp', 'Su', 'Sg', 'Ba', 'Bgr']
mRMR TOP-10
['Bp', 'Su', 'Sg', 'Ba', 'Bgr', 'Ane', 'Bu', 'Pcv', 'Sc', 'Rbcc']
mRMR TOP-15
['Bp', 'Su', 'Sg', 'Ba', 'Bgr', 'Ane', 'Bu', 'Pcv', 'Sc', 'Rbcc', 'Dm', 'Appet', 'Pcc', 'Pe', 'Al']
mRMR TOP-20
['Bp', 'Su', 'Sg', 'Ba', 'Bgr', 'Ane', 'Bu', 'Pcv', 'Sc', 'Rbcc', 'Dm', 'Appet', 'Pcc', 'Pe', 'Al', 'Pot', 'Wbcc', 'Htn', 'Cad', 'Rbc']


In [11]:
# =============================================================================
# STEP 9: DOWNSTREAM MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print(list(models.keys()))


['Logistic Regression', 'Random Forest', 'XGBoost']


In [12]:
# =============================================================================
# STEP 10: mRMR BASELINE MODEL EVALUATION
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

fixed_mrmr_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = mrmr_results[top_k]["X_train"]
    Xte_selected = mrmr_results[top_k]["X_test"]
    selected_features = mrmr_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_mrmr_rows.append({
            "Method": "mRMR",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_mrmr_results_df = pd.DataFrame(fixed_mrmr_rows)
display(fixed_mrmr_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,mRMR,5,Logistic Regression,"Bp, Su, Sg, Ba, Bgr",0.5404,0.5455,0.6471,0.5919,0.5828
1,mRMR,5,Random Forest,"Bp, Su, Sg, Ba, Bgr",0.5505,0.5730,0.5000,0.5340,0.5805
2,mRMR,5,XGBoost,"Bp, Su, Sg, Ba, Bgr",0.5657,0.5800,0.5686,0.5743,0.5416
3,mRMR,10,Logistic Regression,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc",0.5556,0.5625,0.6176,0.5888,0.5901
4,mRMR,10,Random Forest,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc",0.4899,0.5053,0.4706,0.4873,0.5075
5,mRMR,10,XGBoost,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc",0.5152,0.5312,0.5000,0.5152,0.5066
6,mRMR,15,Logistic Regression,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5556,0.5660,0.5882,0.5769,0.5684
7,mRMR,15,Random Forest,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5303,0.5474,0.5098,0.5279,0.5233
8,mRMR,15,XGBoost,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5253,0.5392,0.5392,0.5392,0.5413
9,mRMR,20,Logistic Regression,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5505,0.5596,0.5980,0.5782,0.5751


In [13]:
# =============================================================================
# STEP 11: SAVE mRMR BASELINE
# =============================================================================

baseline_dir = Path("../../../results/kidney_disease/baseline")
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_path = baseline_dir / "mrmr_baseline_results.csv"
fixed_mrmr_results_df.to_csv(baseline_path, index=False)

print(f"Saved: {baseline_path.resolve()}")


Saved: C:\Users\johnm\msc_research\results\kidney_disease\baseline\mrmr_baseline_results.csv


# 2. DODA Clinical Re-ranking

### Correct experimental order

```text
24 original predictors
        ↓
mRMR ranking for all 24
        ↓
statistical ranking
        +
pre-specified clinical weights
        ↓
DODA RankFusion
        ↓
final Top-K
```

DODA must **not** receive an already-truncated mRMR Top-K set. Otherwise, clinically prioritized features outside that preliminary Top-K would be impossible to recover.


In [14]:
# =============================================================================
# STEP 12: INSTALL / IMPORT DODA
# =============================================================================

# Run the following installation once if DODA is not installed:
# %pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

import doda

from doda import DODASelector
from doda.adapters import SklearnAdapter
from doda.knowledge.providers import JSONProvider
from doda.fusion import RankFusion

print("DODA:", doda.__file__)

print("Provider:", JSONProvider)
print("Fusion:", RankFusion)

DODA: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\__init__.py
Provider: <class 'doda.knowledge.providers.json_provider.JSONProvider'>
Fusion: <class 'doda.fusion.rank.RankFusion'>


In [15]:
# =============================================================================
# STEP 13: LOCATE AND VALIDATE CLINICAL WEIGHTS
# =============================================================================

from pathlib import Path
from doda.knowledge.providers import JSONProvider

WEIGHTS_FILENAME = "BD_KDD_clinical_weights.json"

search_roots = [Path.cwd(), *Path.cwd().parents]
matches = []

for root in search_roots:
    try:
        matches.extend(root.rglob(WEIGHTS_FILENAME))
    except (PermissionError, OSError):
        continue

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{WEIGHTS_FILENAME} was not found. "
        "Place it in the repository and rerun this cell."
    )

WEIGHTS_PATH = matches[0]

print(f"Using clinical weights: {WEIGHTS_PATH.resolve()}")

# Use the standard DODA JSON provider
provider = JSONProvider(str(WEIGHTS_PATH))

# Validate coverage using the flat JSON directly
import json

with open(WEIGHTS_PATH, "r", encoding="utf-8") as f:
    clinical_weights = json.load(f)

missing_weights = sorted(
    set(X.columns) - set(clinical_weights)
)

extra_weights = sorted(
    set(clinical_weights) - set(X.columns)
)

print(f"Clinical weights: {len(clinical_weights)}")
print(f"Predictors:       {len(X.columns)}")
print(f"Missing weights:  {missing_weights}")
print(f"Extra weights:    {extra_weights}")

assert not missing_weights
assert not extra_weights

Using clinical weights: C:\Users\johnm\msc_research\config\clinical_weights\BD_KDD_clinical_weights.json
Clinical weights: 24
Predictors:       24
Missing weights:  []
Extra weights:    []


In [16]:
# =============================================================================
# STEP 14: DODA — mRMR SCORE ALL FEATURES, THEN APPLY TOP-K
# =============================================================================

class MRMRAdapter:
    """
    Adapter that runs mRMR on all available features and exposes the
    complete ranking as normalized operator importance scores.
    """

    def __init__(self):
        self.importance = {}

    def fit(self, X, y):
        X_local = pd.DataFrame(X).copy()
        X_local.columns = list(X.columns)
        X_local = X_local.reset_index(drop=True)
        y_local = pd.Series(y).reset_index(drop=True)

        ranked_features = mrmr_classif(
            X=X_local,
            y=y_local,
            K=X_local.shape[1]
        )

        n_features = len(ranked_features)

        self.importance = {
            feature: (n_features - rank) / n_features
            for rank, feature in enumerate(ranked_features)
        }

        return self

    def get_importance(self):
        return self.importance

    def get_name(self):
        return "mRMR"


rank_doda_results = {}

for top_k in TOP_K_VALUES:

    mrmr_operator = MRMRAdapter()

    doda_selector = DODASelector(
        operators=[mrmr_operator],
        provider=provider,
        fusion=RankFusion(),
        top_k=top_k
    )

    # Fit on the scaled training data only.
    doda_selector.fit(
        X_train_mrmr_scaled,
        y_train
    )

    selected_features = list(
        doda_selector.get_selected_features()
    )

    rank_doda_results[top_k] = {
        "selector": doda_selector,
        "features": selected_features,
        "X_train": X_train[selected_features].copy(),
        "X_test": X_test[selected_features].copy()
    }

    print("=" * 70)
    print(f"DODA TOP-{top_k}")
    print("=" * 70)
    print(selected_features)
    print(f"Number selected: {len(selected_features)}")

    assert len(selected_features) == top_k


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x0000019B7D6DBA10>
DODA Pipeline Started
Running: mRMR


100%|██████████| 24/24 [00:07<00:00,  3.34it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 'Htn': 0.2916666666666667, 'Cad': 0.25, 'Rbc': 0.20833333333333334, 'Hemo': 0.16666666666666666, 'Age': 0.125, 'Sod': 0.08333333333333333, 'Pc': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 

100%|██████████| 24/24 [00:06<00:00,  3.52it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 'Htn': 0.2916666666666667, 'Cad': 0.25, 'Rbc': 0.20833333333333334, 'Hemo': 0.16666666666666666, 'Age': 0.125, 'Sod': 0.08333333333333333, 'Pc': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 

100%|██████████| 24/24 [00:07<00:00,  3.37it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 'Htn': 0.2916666666666667, 'Cad': 0.25, 'Rbc': 0.20833333333333334, 'Hemo': 0.16666666666666666, 'Age': 0.125, 'Sod': 0.08333333333333333, 'Pc': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 

100%|██████████| 24/24 [00:06<00:00,  3.47it/s]

Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 'Htn': 0.2916666666666667, 'Cad': 0.25, 'Rbc': 0.20833333333333334, 'Hemo': 0.16666666666666666, 'Age': 0.125, 'Sod': 0.08333333333333333, 'Pc': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Ba': 0.875, 'Bgr': 0.8333333333333334, 'Ane': 0.7916666666666666, 'Bu': 0.75, 'Pcv': 0.7083333333333334, 'Sc': 0.6666666666666666, 'Rbcc': 0.625, 'Dm': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Pcc': 0.5, 'Pe': 0.4583333333333333, 'Al': 0.4166666666666667, 'Pot': 0.375, 'Wbcc': 0.3333333333333333, 

In [17]:
# =============================================================================
# STEP 15: INSPECT DODA RANK CHANGES
# =============================================================================

INSPECTION_TOP_K = 10

selector = rank_doda_results[INSPECTION_TOP_K]["selector"]

print("=" * 70)
print(f"DODA RANK COMPARISON — TOP-{INSPECTION_TOP_K}")
print("=" * 70)

rank_comparison = selector.compare_scores()

display(rank_comparison)

print("\nSelected features:")
print(rank_doda_results[INSPECTION_TOP_K]["features"])


DODA RANK COMPARISON — TOP-10

MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      Bp          1              6           1            0
1      Sg          3              4           2            1
2      Bu          7              3           3            4
3      Sc          9              1           4            5
4      Su          2             15           5           -3
5     Bgr          5              9           6           -1
6     Ane          6             10           7           -1
7     Pcv          8             11           8            0
8      Al         15              2           9            6
9      Ba          4             21          10           -6


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,Bp,1,1.0000,6,0.80,1,0.0320,0
1,Sg,3,0.9167,4,0.85,2,0.0317,1
2,Bu,7,0.7500,3,0.90,3,0.0311,4
3,Sc,9,0.6667,1,1.00,4,0.0309,5
4,Su,2,0.9583,15,0.55,5,0.0308,-3
5,Bgr,5,0.8333,9,0.70,6,0.0305,-1
6,Ane,6,0.7917,10,0.70,7,0.0303,-1
7,Pcv,8,0.7083,11,0.70,8,0.0299,0
8,Al,15,0.4167,2,1.00,9,0.0297,6
9,Ba,4,0.8750,21,0.40,10,0.0297,-6



Selected features:
['Bp', 'Sg', 'Bu', 'Sc', 'Su', 'Bgr', 'Ane', 'Pcv', 'Al', 'Ba']


In [18]:
# =============================================================================
# STEP 15B: EXPLICIT mRMR vs DODA RANK-SHIFT TABLE
# =============================================================================
# This table separates:
#   - statistical mRMR rank
#   - DODA final rank
#   - clinical weight
#   - rank movement
#
# Positive Rank_Change means the feature moved upward.
# Example: mRMR rank 10 -> DODA rank 4 = +6
#
# IMPORTANT:
# The selector is explicitly retrieved from rank_doda_results
# for the selected Top-K value. This avoids relying on a
# previously defined 'selector' variable.

# -------------------------------------------------------------------------
# Select which DODA Top-K experiment to inspect
# -------------------------------------------------------------------------

k = 10

if k not in rank_doda_results:
    raise KeyError(
        f"Top-K={k} was not found in rank_doda_results. "
        f"Available values: {list(rank_doda_results.keys())}"
    )

selector = rank_doda_results[k]["selector"]


# -------------------------------------------------------------------------
# mRMR / mathematical ranking
# -------------------------------------------------------------------------

math_rank = pd.DataFrame(
    selector.math_scores_.items(),
    columns=[
        "Feature",
        "mRMR_Normalized_Score"
    ]
)

math_rank["mRMR_Rank"] = (
    math_rank["mRMR_Normalized_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# DODA final ranking
# -------------------------------------------------------------------------

final_rank = pd.DataFrame(
    selector.final_scores_.items(),
    columns=[
        "Feature",
        "DODA_Final_Score"
    ]
)

final_rank["DODA_Rank"] = (
    final_rank["DODA_Final_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# Combine rankings
# -------------------------------------------------------------------------

rank_shift_df = (
    math_rank[
        [
            "Feature",
            "mRMR_Normalized_Score",
            "mRMR_Rank"
        ]
    ]
    .merge(
        final_rank[
            [
                "Feature",
                "DODA_Final_Score",
                "DODA_Rank"
            ]
        ],
        on="Feature",
        how="inner"
    )
)


# -------------------------------------------------------------------------
# Add clinical weights
# -------------------------------------------------------------------------

rank_shift_df["Clinical_Weight"] = (
    rank_shift_df["Feature"]
    .map(selector.clinical_weights_)
)


# -------------------------------------------------------------------------
# Calculate rank movement
# -------------------------------------------------------------------------
# Positive value  = moved upward
# Negative value  = moved downward
# Zero             = no rank movement

rank_shift_df["Rank_Change"] = (
    rank_shift_df["mRMR_Rank"]
    - rank_shift_df["DODA_Rank"]
)


# -------------------------------------------------------------------------
# Sort by DODA final rank
# -------------------------------------------------------------------------

rank_shift_df = (
    rank_shift_df
    .sort_values(
        by="DODA_Rank"
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

print("=" * 80)
print(f"mRMR vs DODA RANK-SHIFT ANALYSIS — TOP-{k}")
print("=" * 80)

display(rank_shift_df)

mRMR vs DODA RANK-SHIFT ANALYSIS — TOP-10


,Feature,mRMR_Normalized_Score,mRMR_Rank,DODA_Final_Score,DODA_Rank,Clinical_Weight,Rank_Change
0,Bp,1.000000,1,0.032018,1,0.80,0
1,Sg,0.916667,3,0.031746,2,0.85,1
2,Bu,0.750000,7,0.031054,3,0.90,4
3,Sc,0.666667,9,0.030886,4,1.00,5
4,Su,0.958333,2,0.030835,5,0.55,-3
5,Bgr,0.833333,5,0.030536,6,0.70,-1
6,Ane,0.791667,6,0.030303,7,0.70,-1
7,Pcv,0.708333,8,0.029857,8,0.70,0
8,Al,0.416667,15,0.029727,9,1.00,6
9,Ba,0.875000,4,0.029710,10,0.40,-6


In [19]:
# =============================================================================
# STEP 16: INSPECT CLINICAL WEIGHTS
# =============================================================================

clinical_weight_table = pd.DataFrame({
    "Feature": list(selector.clinical_weights_.keys()),
    "Clinical_Weight": list(selector.clinical_weights_.values())
}).sort_values(
    "Clinical_Weight",
    ascending=False
).reset_index(drop=True)

display(clinical_weight_table)


,Feature,Clinical_Weight
0,Al,1.00
1,Sc,1.00
2,Bu,0.90
3,Sg,0.85
4,Htn,0.85
5,Dm,0.80
6,Bp,0.80
7,Hemo,0.75
8,Bgr,0.70
9,Ane,0.70


In [20]:
# =============================================================================
# STEP 17: DODA MODEL EVALUATION — FIXED SPLIT
# =============================================================================

fixed_doda_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rank_doda_results[top_k]["X_train"]
    Xte_selected = rank_doda_results[top_k]["X_test"]
    selected_features = rank_doda_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_doda_rows.append({
            "Method": "DODA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_doda_results_df = pd.DataFrame(fixed_doda_rows)
display(fixed_doda_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,DODA,5,Logistic Regression,"Bp, Sg, Bu, Sc, Su",0.5505,0.5575,0.6176,0.5860,0.5524
1,DODA,5,Random Forest,"Bp, Sg, Bu, Sc, Su",0.4697,0.4848,0.4706,0.4776,0.4526
2,DODA,5,XGBoost,"Bp, Sg, Bu, Sc, Su",0.5000,0.5146,0.5196,0.5171,0.4922
3,DODA,10,Logistic Regression,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba",0.5354,0.5446,0.5980,0.5701,0.5797
4,DODA,10,Random Forest,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba",0.5253,0.5392,0.5392,0.5392,0.5447
5,DODA,10,XGBoost,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba",0.5303,0.5437,0.5490,0.5463,0.5350
6,DODA,15,Logistic Regression,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba, Dm,...",0.5657,0.5727,0.6176,0.5943,0.5856
7,DODA,15,Random Forest,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba, Dm,...",0.4949,0.5104,0.4804,0.4949,0.5362
8,DODA,15,XGBoost,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba, Dm,...",0.4697,0.4865,0.5294,0.5070,0.4970
9,DODA,20,Logistic Regression,"Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba, Dm,...",0.5707,0.5794,0.6078,0.5933,0.5763


In [21]:
# =============================================================================
# STEP 18: FIXED-SPLIT mRMR vs DODA COMPARISON
# =============================================================================

fixed_comparison_df = pd.concat(
    [fixed_mrmr_results_df, fixed_doda_results_df],
    ignore_index=True
)

display(fixed_comparison_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,mRMR,5,Logistic Regression,"Bp, Su, Sg, Ba, Bgr",0.5404,0.5455,0.6471,0.5919,0.5828
1,mRMR,5,Random Forest,"Bp, Su, Sg, Ba, Bgr",0.5505,0.5730,0.5000,0.5340,0.5805
2,mRMR,5,XGBoost,"Bp, Su, Sg, Ba, Bgr",0.5657,0.5800,0.5686,0.5743,0.5416
3,mRMR,10,Logistic Regression,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc",0.5556,0.5625,0.6176,0.5888,0.5901
4,mRMR,10,Random Forest,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc",0.4899,0.5053,0.4706,0.4873,0.5075
5,mRMR,10,XGBoost,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc",0.5152,0.5312,0.5000,0.5152,0.5066
6,mRMR,15,Logistic Regression,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5556,0.5660,0.5882,0.5769,0.5684
7,mRMR,15,Random Forest,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5303,0.5474,0.5098,0.5279,0.5233
8,mRMR,15,XGBoost,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5253,0.5392,0.5392,0.5392,0.5413
9,mRMR,20,Logistic Regression,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...",0.5505,0.5596,0.5980,0.5782,0.5751


In [22]:
# =============================================================================
# STEP 19: COMPARE FEATURE SETS ON THE FIXED SPLIT
# =============================================================================

fixed_feature_comparison = []

for top_k in TOP_K_VALUES:

    mrmr_set = set(mrmr_results[top_k]["features"])
    doda_set = set(rank_doda_results[top_k]["features"])

    intersection = mrmr_set & doda_set
    union = mrmr_set | doda_set

    jaccard = len(intersection) / len(union)

    fixed_feature_comparison.append({
        "Top_K": top_k,
        "mRMR_Features": ", ".join(mrmr_results[top_k]["features"]),
        "DODA_Features": ", ".join(rank_doda_results[top_k]["features"]),
        "Same_Feature_Set": mrmr_set == doda_set,
        "Overlap_Count": len(intersection),
        "Jaccard": jaccard,
        "Features_Changed": top_k - len(intersection)
    })

fixed_feature_comparison_df = pd.DataFrame(fixed_feature_comparison)

display(fixed_feature_comparison_df)


,Top_K,mRMR_Features,DODA_Features,Same_Feature_Set,Overlap_Count,Jaccard,Features_Changed
0,5,"Bp, Su, Sg, Ba, Bgr","Bp, Sg, Bu, Sc, Su",False,3,0.428571,2
1,10,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc","Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba",False,9,0.818182,1
2,15,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...","Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba, Dm,...",False,13,0.764706,2
3,20,"Bp, Su, Sg, Ba, Bgr, Ane, Bu, Pcv, Sc, Rbcc, D...","Bp, Sg, Bu, Sc, Su, Bgr, Ane, Pcv, Al, Ba, Dm,...",False,19,0.904762,1


In [23]:
# =============================================================================
# STEP 20: SAVE FIXED-SPLIT DODA RESULTS
# =============================================================================

doda_dir = Path("../../../results/kidney_disease/doda")
doda_dir.mkdir(parents=True, exist_ok=True)

fixed_doda_results_df.to_csv(
    doda_dir / "mrmr_doda_fixed_split_results.csv",
    index=False
)

fixed_feature_comparison_df.to_csv(
    doda_dir / "mrmr_doda_fixed_split_feature_comparison.csv",
    index=False
)

print(f"Saved results to: {doda_dir.resolve()}")


Saved results to: C:\Users\johnm\msc_research\results\kidney_disease\doda


# 3.1 Five-Fold Stratified Cross-Validation

The fixed 80/20 split is followed by a separate **5-fold stratified cross-validation** experiment.

Feature selection is refit inside each training fold. DODA also receives all features and performs the final Top-K selection inside the training fold. Logistic Regression scaling is fitted only on the corresponding training fold.

The DODA fusion method remains **RankFusion**, matching the current BD-KDD pipeline.

In [24]:
# =============================================================================
# STEP 21: 5-FOLD CV CONFIGURATION
# =============================================================================
from sklearn.model_selection import StratifiedKFold

CV5_SPLITS = 5
CV5_RANDOM_STATE = 42
cv5 = StratifiedKFold(n_splits=CV5_SPLITS, shuffle=True, random_state=CV5_RANDOM_STATE)
cv5_results = []

print("5-fold CV configuration loaded.")
print("Splits:", CV5_SPLITS)
print("Top-K values:", TOP_K_VALUES)
print("Fusion: RankFusion")


5-fold CV configuration loaded.
Splits: 5
Top-K values: [5, 10, 15, 20]
Fusion: RankFusion


In [25]:
# =============================================================================
# STEP 22: 5-FOLD CV — mRMR vs DODA
# =============================================================================

# Reset results before running the CV.
# This prevents duplicate rows if the cell is executed more than once.
cv5_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv5.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    # -------------------------------------------------------------------------
    # Scale ONLY using the training-fold statistics before mRMR selection.
    # -------------------------------------------------------------------------

    fold_scaler = StandardScaler()

    X_train_fold_scaled = pd.DataFrame(
        fold_scaler.fit_transform(X_train_fold),
        columns=X_train_fold.columns,
        index=X_train_fold.index
    )

    X_val_fold_scaled = pd.DataFrame(
        fold_scaler.transform(X_val_fold),
        columns=X_val_fold.columns,
        index=X_val_fold.index
    )

    # -------------------------------------------------------------------------
    # mRMR ranking is learned ONLY from the training fold.
    # -------------------------------------------------------------------------

    X_mrmr_fold = X_train_fold_scaled.reset_index(drop=True)
    y_mrmr_fold = y_train_fold.reset_index(drop=True)

    mrmr_fold_ranked = mrmr_classif(
        X=X_mrmr_fold,
        y=y_mrmr_fold,
        K=X_mrmr_fold.shape[1]
    )

    # -------------------------------------------------------------------------
    # Evaluate each Top-K
    # -------------------------------------------------------------------------

    for top_k in TOP_K_VALUES:

        mrmr_features = mrmr_fold_ranked[:top_k]

        fold_doda = DODASelector(
            operators=[MRMRAdapter()],
            provider=JSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold_scaled,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        feature_sets = {
            "mRMR": mrmr_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv5_results.append({
                    "Fold": fold,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Selected_Features": ", ".join(selected_features),
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(y_val_fold, y_pred, zero_division=0),
                    "Recall": recall_score(y_val_fold, y_pred, zero_division=0),
                    "F1": f1_score(y_val_fold, y_pred, zero_division=0),
                    "ROC_AUC": roc_auc_score(y_val_fold, y_prob)
                })

cv5_results_df = pd.DataFrame(cv5_results)

expected_rows = CV5_SPLITS * len(TOP_K_VALUES) * 2 * len(models)

print(f"Expected rows: {expected_rows}")
print(f"Actual rows:   {len(cv5_results_df)}")

assert len(cv5_results_df) == expected_rows

display(cv5_results_df.round(4))


100%|██████████| 24/24 [00:06<00:00,  3.57it/s]


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x0000019B7D73A490>
DODA Pipeline Started
Running: mRMR


100%|██████████| 24/24 [00:06<00:00,  3.65it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Ba': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Pot': 0.875, 'Hemo': 0.8333333333333334, 'Wbcc': 0.7916666666666666, 'Su': 0.75, 'Rbcc': 0.7083333333333334, 'Age': 0.6666666666666666, 'Bgr': 0.625, 'Sc': 0.5833333333333334, 'Sod': 0.5416666666666666, 'Pc': 0.5, 'Appet': 0.4583333333333333, 'Rbc': 0.4166666666666667, 'Htn': 0.375, 'Ane': 0.3333333333333333, 'Pcv': 0.2916666666666667, 'Cad': 0.25, 'Pe': 0.20833333333333334, 'Al': 0.16666666666666666, 'Pcc': 0.125, 'Dm': 0.08333333333333333, 'Bu': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Ba': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Pot': 0.875, 'Hemo': 0.8333333333333334, 'Wbcc': 0.7916666666666666, 'Su': 0.75, 'Rbcc': 0.7083333333333334, 'Age': 0.6666666666666666, 'Bgr': 0.625, 'Sc': 0.5833333333333334, 'Sod': 0.5416666666666666, 'Pc': 0.5, 'Appet': 0.4583333333333333, 'Rbc': 0.4166666666666667, 'Htn': 0.375, 'Ane': 0.33333333333333

100%|██████████| 24/24 [00:06<00:00,  3.64it/s]


...

100%|██████████| 24/24 [00:07<00:00,  3.42it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Pcv': 0.9166666666666666, 'Bu': 0.875, 'Ba': 0.8333333333333334, 'Pc': 0.7916666666666666, 'Ane': 0.75, 'Rbcc': 0.7083333333333334, 'Appet': 0.6666666666666666, 'Htn': 0.625, 'Sc': 0.5833333333333334, 'Sg': 0.5416666666666666, 'Pot': 0.5, 'Bgr': 0.4583333333333333, 'Pcc': 0.4166666666666667, 'Sod': 0.375, 'Al': 0.3333333333333333, 'Pe': 0.2916666666666667, 'Rbc': 0.25, 'Hemo': 0.20833333333333334, 'Wbcc': 0.16666666666666666, 'Dm': 0.125, 'Cad': 0.08333333333333333, 'Age': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Pcv': 0.9166666666666666, 'Bu': 0.875, 'Ba': 0.8333333333333334, 'Pc': 0.7916666666666666, 'Ane': 0.75, 'Rbcc': 0.7083333333333334, 'Appet': 0.6666666666666666, 'Htn': 0.625, 'Sc': 0.5833333333333334, 'Sg': 0.5416666666666666, 'Pot': 0.5, 'Bgr': 0.4583333333333333, 'Pcc': 0.4166666666666667, 'Sod': 0.375, 'Al': 0.3333333333333333, 

100%|██████████| 24/24 [00:06<00:00,  3.61it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Pcv': 0.9166666666666666, 'Bu': 0.875, 'Ba': 0.8333333333333334, 'Pc': 0.7916666666666666, 'Ane': 0.75, 'Rbcc': 0.7083333333333334, 'Appet': 0.6666666666666666, 'Htn': 0.625, 'Sc': 0.5833333333333334, 'Sg': 0.5416666666666666, 'Pot': 0.5, 'Bgr': 0.4583333333333333, 'Pcc': 0.4166666666666667, 'Sod': 0.375, 'Al': 0.3333333333333333, 'Pe': 0.2916666666666667, 'Rbc': 0.25, 'Hemo': 0.20833333333333334, 'Wbcc': 0.16666666666666666, 'Dm': 0.125, 'Cad': 0.08333333333333333, 'Age': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Pcv': 0.9166666666666666, 'Bu': 0.875, 'Ba': 0.8333333333333334, 'Pc': 0.7916666666666666, 'Ane': 0.75, 'Rbcc': 0.7083333333333334, 'Appet': 0.6666666666666666, 'Htn': 0.625, 'Sc': 0.5833333333333334, 'Sg': 0.5416666666666666, 'Pot': 0.5, 'Bgr': 0.4583333333333333, 'Pcc': 0.4166666666666667, 'Sod': 0.375, 'Al': 0.3333333333333333, 

,Fold,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,mRMR,5,Logistic Regression,"Bp, Ba, Sg, Pot, Hemo",0.4899,0.5000,0.5347,0.5167,0.5135
1,1,mRMR,5,Random Forest,"Bp, Ba, Sg, Pot, Hemo",0.4798,0.4907,0.5248,0.5072,0.5059
2,1,mRMR,5,XGBoost,"Bp, Ba, Sg, Pot, Hemo",0.5505,0.5556,0.5941,0.5742,0.5326
3,1,DODA,5,Logistic Regression,"Bp, Sg, Hemo, Pot, Sc",0.5051,0.5146,0.5248,0.5196,0.5076
4,1,DODA,5,Random Forest,"Bp, Sg, Hemo, Pot, Sc",0.5354,0.5455,0.5347,0.5400,0.5405
...,...,...,...,...,...,...,...,...,...,...
115,5,mRMR,20,Random Forest,"Bp, Su, Pcv, Bu, Ba, Pc, Ane, Rbcc, Appet, Htn...",0.5228,0.5310,0.5941,0.5607,0.5179
116,5,mRMR,20,XGBoost,"Bp, Su, Pcv, Bu, Ba, Pc, Ane, Rbcc, Appet, Htn...",0.5127,0.5225,0.5743,0.5472,0.4830
117,5,DODA,20,Logistic Regression,"Bp, Bu, Pcv, Su, Sc, Htn, Ane, Sg, Pc, Ba, Rbc...",0.5178,0.5263,0.5941,0.5581,0.5259
118,5,DODA,20,Random Forest,"Bp, Bu, Pcv, Su, Sc, Htn, Ane, Sg, Pc, Ba, Rbc...",0.5076,0.5175,0.5842,0.5488,0.4818


In [26]:
# =============================================================================
# STEP 23: 5-FOLD CV PERFORMANCE SUMMARY
# =============================================================================
cv5_summary_df = (
    cv5_results_df.groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)
cv5_summary_df.columns = [
    "Method", "Top_K", "Model", "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD", "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD", "ROC_AUC_Mean", "ROC_AUC_STD"
]
display(cv5_summary_df.round(4))

cv5_dir = Path("../../../results/kidney_disease/cv")
cv5_dir.mkdir(parents=True, exist_ok=True)
cv5_results_df.to_csv(cv5_dir / "mrmr_doda_5fold_results.csv", index=False)
cv5_summary_df.to_csv(cv5_dir / "mrmr_doda_5fold_summary.csv", index=False)


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,DODA,5,Logistic Regression,0.5182,0.0261,0.5272,0.0213,0.5859,0.0599,0.5542,0.0353,0.5141,0.0208
1,DODA,5,Random Forest,0.5121,0.0245,0.5257,0.0244,0.5307,0.0265,0.5274,0.0142,0.4961,0.0340
2,DODA,5,XGBoost,0.5151,0.0342,0.5280,0.0332,0.5346,0.0524,0.5303,0.0361,0.5107,0.0547
3,DODA,10,Logistic Regression,0.5304,0.0121,0.5392,0.0084,0.5800,0.0344,0.5586,0.0201,0.5231,0.0192
4,DODA,10,Random Forest,0.5009,0.0486,0.5168,0.0523,0.5306,0.0319,0.5223,0.0315,0.5048,0.0515
5,DODA,10,XGBoost,0.5111,0.0396,0.5229,0.0362,0.5464,0.0362,0.5343,0.0354,0.5032,0.0478
6,DODA,15,Logistic Regression,0.5243,0.0092,0.5343,0.0069,0.5681,0.0130,0.5507,0.0094,0.5239,0.0176
7,DODA,15,Random Forest,0.5162,0.0285,0.5279,0.0264,0.5484,0.0366,0.5376,0.0271,0.5142,0.0407
8,DODA,15,XGBoost,0.5020,0.0597,0.5136,0.0540,0.5268,0.0757,0.5196,0.0631,0.5112,0.0646
9,DODA,20,Logistic Regression,0.5202,0.0093,0.5302,0.0080,0.5740,0.0284,0.5509,0.0137,0.5250,0.0177


# 3.2 Robustness: 5×5 Repeated Stratified Cross-Validation

The fixed split provides a reproducible initial comparison.

For robustness, feature selection is **recomputed inside every training fold**. This prevents information from the validation fold from influencing mRMR or DODA.

There are:

- 5 folds
- 5 repeats
- 25 train/validation runs
- 4 Top-K budgets
- 2 feature-selection methods
- 3 downstream models

This produces 600 model-evaluation rows.


In [27]:
from mrmr import mrmr_classif

# =============================================================================
# STEP 24: REPEATED CV CONFIGURATION
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold

TOP_K_VALUES = [5, 10, 15, 20]

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

print(f"Top-K values: {TOP_K_VALUES}")
print(f"CV runs: {N_SPLITS * N_REPEATS}")


Top-K values: [5, 10, 15, 20]
CV runs: 25


In [28]:
# =============================================================================
# STEP 25: 5×5 REPEATED CV — mRMR vs DODA
# =============================================================================

cv_results = []

for run_id, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    print(f"Run {run_id:02d}/{N_SPLITS * N_REPEATS}")

    fold_scaler = StandardScaler()

    X_train_fold_scaled = pd.DataFrame(
        fold_scaler.fit_transform(X_train_fold),
        columns=X_train_fold.columns,
        index=X_train_fold.index
    )

    X_val_fold_scaled = pd.DataFrame(
        fold_scaler.transform(X_val_fold),
        columns=X_val_fold.columns,
        index=X_val_fold.index
    )

    X_mrmr_fold = X_train_fold_scaled.reset_index(drop=True)
    y_mrmr_fold = y_train_fold.reset_index(drop=True)

    mrmr_fold_ranked = mrmr_classif(
        X=X_mrmr_fold,
        y=y_mrmr_fold,
        K=X_mrmr_fold.shape[1]
    )

    for top_k in TOP_K_VALUES:

        mrmr_features = mrmr_fold_ranked[:top_k]

        fold_doda = DODASelector(
            operators=[MRMRAdapter()],
            provider=JSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold_scaled,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        feature_sets = {
            "mRMR": mrmr_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv_results.append({
                    "Run": run_id,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Selected_Features": ", ".join(selected_features),
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(y_val_fold, y_pred, zero_division=0),
                    "Recall": recall_score(y_val_fold, y_pred, zero_division=0),
                    "F1": f1_score(y_val_fold, y_pred, zero_division=0),
                    "ROC_AUC": roc_auc_score(y_val_fold, y_prob)
                })

cv_results_df = pd.DataFrame(cv_results)

expected_rows = N_SPLITS * N_REPEATS * len(TOP_K_VALUES) * 2 * len(models)

print(f"Expected repeated-CV rows: {expected_rows}")
print(f"Actual repeated-CV rows:   {len(cv_results_df)}")

assert len(cv_results_df) == expected_rows

display(cv_results_df.round(4))


Run 01/25


100%|██████████| 24/24 [00:06<00:00,  3.49it/s]


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x0000019B7FA01EB0>
DODA Pipeline Started
Running: mRMR


100%|██████████| 24/24 [00:06<00:00,  3.59it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Ba': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Pot': 0.875, 'Hemo': 0.8333333333333334, 'Wbcc': 0.7916666666666666, 'Su': 0.75, 'Rbcc': 0.7083333333333334, 'Age': 0.6666666666666666, 'Bgr': 0.625, 'Sc': 0.5833333333333334, 'Sod': 0.5416666666666666, 'Pc': 0.5, 'Appet': 0.4583333333333333, 'Rbc': 0.4166666666666667, 'Htn': 0.375, 'Ane': 0.3333333333333333, 'Pcv': 0.2916666666666667, 'Cad': 0.25, 'Pe': 0.20833333333333334, 'Al': 0.16666666666666666, 'Pcc': 0.125, 'Dm': 0.08333333333333333, 'Bu': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Ba': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Pot': 0.875, 'Hemo': 0.8333333333333334, 'Wbcc': 0.7916666666666666, 'Su': 0.75, 'Rbcc': 0.7083333333333334, 'Age': 0.6666666666666666, 'Bgr': 0.625, 'Sc': 0.5833333333333334, 'Sod': 0.5416666666666666, 'Pc': 0.5, 'Appet': 0.4583333333333333, 'Rbc': 0.4166666666666667, 'Htn': 0.375, 'Ane': 0.33333333333333

...

100%|██████████| 24/24 [00:07<00:00,  3.28it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Bu': 0.875, 'Pot': 0.8333333333333334, 'Pcv': 0.7916666666666666, 'Rbcc': 0.75, 'Ane': 0.7083333333333334, 'Ba': 0.6666666666666666, 'Sc': 0.625, 'Al': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Bgr': 0.5, 'Hemo': 0.4583333333333333, 'Age': 0.4166666666666667, 'Pe': 0.375, 'Wbcc': 0.3333333333333333, 'Cad': 0.2916666666666667, 'Htn': 0.25, 'Dm': 0.20833333333333334, 'Sod': 0.16666666666666666, 'Pc': 0.125, 'Pcc': 0.08333333333333333, 'Rbc': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Bu': 0.875, 'Pot': 0.8333333333333334, 'Pcv': 0.7916666666666666, 'Rbcc': 0.75, 'Ane': 0.7083333333333334, 'Ba': 0.6666666666666666, 'Sc': 0.625, 'Al': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Bgr': 0.5, 'Hemo': 0.4583333333333333, 'Age': 0.4166666666666667, 'Pe': 0.375, 'Wbcc': 0.3333333333333333

100%|██████████| 24/24 [00:06<00:00,  3.51it/s]


Operator Scores:
{'mRMR_1': {'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Bu': 0.875, 'Pot': 0.8333333333333334, 'Pcv': 0.7916666666666666, 'Rbcc': 0.75, 'Ane': 0.7083333333333334, 'Ba': 0.6666666666666666, 'Sc': 0.625, 'Al': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Bgr': 0.5, 'Hemo': 0.4583333333333333, 'Age': 0.4166666666666667, 'Pe': 0.375, 'Wbcc': 0.3333333333333333, 'Cad': 0.2916666666666667, 'Htn': 0.25, 'Dm': 0.20833333333333334, 'Sod': 0.16666666666666666, 'Pc': 0.125, 'Pcc': 0.08333333333333333, 'Rbc': 0.041666666666666664}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'Bp': 1.0, 'Su': 0.9583333333333334, 'Sg': 0.9166666666666666, 'Bu': 0.875, 'Pot': 0.8333333333333334, 'Pcv': 0.7916666666666666, 'Rbcc': 0.75, 'Ane': 0.7083333333333334, 'Ba': 0.6666666666666666, 'Sc': 0.625, 'Al': 0.5833333333333334, 'Appet': 0.5416666666666666, 'Bgr': 0.5, 'Hemo': 0.4583333333333333, 'Age': 0.4166666666666667, 'Pe': 0.375, 'Wbcc': 0.3333333333333333

,Run,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,mRMR,5,Logistic Regression,"Bp, Ba, Sg, Pot, Hemo",0.4899,0.5000,0.5347,0.5167,0.5135
1,1,mRMR,5,Random Forest,"Bp, Ba, Sg, Pot, Hemo",0.4798,0.4907,0.5248,0.5072,0.5059
2,1,mRMR,5,XGBoost,"Bp, Ba, Sg, Pot, Hemo",0.5505,0.5556,0.5941,0.5742,0.5326
3,1,DODA,5,Logistic Regression,"Bp, Sg, Hemo, Pot, Sc",0.5051,0.5146,0.5248,0.5196,0.5076
4,1,DODA,5,Random Forest,"Bp, Sg, Hemo, Pot, Sc",0.5354,0.5455,0.5347,0.5400,0.5405
...,...,...,...,...,...,...,...,...,...,...
595,25,mRMR,20,Random Forest,"Bp, Su, Sg, Bu, Pot, Pcv, Rbcc, Ane, Ba, Sc, A...",0.5025,0.5133,0.5743,0.5421,0.5016
596,25,mRMR,20,XGBoost,"Bp, Su, Sg, Bu, Pot, Pcv, Rbcc, Ane, Ba, Sc, A...",0.5127,0.5225,0.5743,0.5472,0.5132
597,25,DODA,20,Logistic Regression,"Bp, Bu, Sg, Su, Sc, Al, Pot, Pcv, Ane, Rbcc, H...",0.5381,0.5424,0.6337,0.5845,0.5154
598,25,DODA,20,Random Forest,"Bp, Bu, Sg, Su, Sc, Al, Pot, Pcv, Ane, Rbcc, H...",0.5381,0.5463,0.5842,0.5646,0.5283


In [29]:
# =============================================================================
# STEP 26: CV PERFORMANCE SUMMARY
# =============================================================================

cv_summary_df = (
    cv_results_df
    .groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)

cv_summary_df.columns = [
    "Method", "Top_K", "Model",
    "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD",
    "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD",
    "ROC_AUC_Mean", "ROC_AUC_STD"
]

display(cv_summary_df.round(4))


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,DODA,5,Logistic Regression,0.5129,0.0362,0.5228,0.0319,0.5768,0.0559,0.5478,0.0397,0.5156,0.0376
1,DODA,5,Random Forest,0.5091,0.0365,0.5207,0.0367,0.5333,0.0612,0.5260,0.0449,0.4965,0.0442
2,DODA,5,XGBoost,0.5044,0.0362,0.5160,0.0347,0.5301,0.0595,0.5223,0.0445,0.5036,0.0433
3,DODA,10,Logistic Regression,0.5180,0.0292,0.5280,0.0260,0.5708,0.0450,0.5481,0.0318,0.5203,0.0359
4,DODA,10,Random Forest,0.4970,0.0378,0.5104,0.0368,0.5278,0.0441,0.5182,0.0352,0.4963,0.0426
5,DODA,10,XGBoost,0.5012,0.0347,0.5133,0.0333,0.5302,0.0499,0.5211,0.0390,0.5011,0.0413
6,DODA,15,Logistic Regression,0.5206,0.0332,0.5309,0.0303,0.5633,0.0455,0.5463,0.0351,0.5239,0.0363
7,DODA,15,Random Forest,0.5045,0.0283,0.5173,0.0269,0.5357,0.0345,0.5258,0.0255,0.5108,0.0378
8,DODA,15,XGBoost,0.5126,0.0390,0.5242,0.0354,0.5393,0.0519,0.5312,0.0413,0.5158,0.0460
9,DODA,20,Logistic Regression,0.5180,0.0311,0.5287,0.0296,0.5598,0.0468,0.5433,0.0342,0.5217,0.0346


In [30]:
# =============================================================================
# STEP 24: SAVE REPEATED-CV RESULTS
# =============================================================================

cv_dir = Path("../../../results/kidney_disease/cv")
cv_dir.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(
    cv_dir / "mrmr_vs_doda_5x5_repeated_cv_results.csv",
    index=False
)

cv_summary_df.to_csv(
    cv_dir / "mrmr_vs_doda_5x5_repeated_cv_summary.csv",
    index=False
)

print(f"Saved to: {cv_dir.resolve()}")


Saved to: C:\Users\johnm\msc_research\results\kidney_disease\cv


# 4. Feature-Selection Stability, Rank/Set Change, and Statistical Comparison

Two different concepts are reported:

- **Feature-set change:** whether the Top-K membership changes.
- **Rank change:** whether feature ordering changes even when membership is retained.

Jaccard similarity evaluates set overlap, so it does not measure ordering. The fixed-split DODA rank comparison is used to inspect ordering changes.


In [31]:
# =============================================================================
# STEP 28: FEATURE-SET COMPARISON ACROSS THE 25 CV RUNS
# =============================================================================

feature_comparison_rows = []

for run_id in range(1, 26):

    for top_k in TOP_K_VALUES:

        mrmr_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "mRMR") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        doda_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        mrmr_set = set(
            mrmr_row["Selected_Features"].split(", ")
        )
        doda_set = set(
            doda_row["Selected_Features"].split(", ")
        )

        overlap = len(mrmr_set & doda_set)
        union = len(mrmr_set | doda_set)

        feature_comparison_rows.append({
            "Run": run_id,
            "Top_K": top_k,
            "Same_Feature_Set": mrmr_set == doda_set,
            "Overlap_Count": overlap,
            "Jaccard": overlap / union,
            "Features_Changed": top_k - overlap
        })

feature_comparison_df = pd.DataFrame(
    feature_comparison_rows
)

display(
    feature_comparison_df
    .groupby("Top_K")
    [["Same_Feature_Set", "Overlap_Count", "Jaccard", "Features_Changed"]]
    .agg(["mean", "std"])
    .round(4)
)


Same_Feature_Set         Overlap_Count         Jaccard          \
                  mean     std          mean     std    mean     std   
Top_K                                                                  
5                 0.04  0.2000          3.72  0.6137  0.6062  0.1491   
10                0.08  0.2769          8.68  0.6904  0.7731  0.1075   
15                0.04  0.2000         13.36  0.7000  0.8060  0.0768   
20                0.04  0.2000         18.36  0.9074  0.8515  0.0764   

      Features_Changed          
                  mean     std  
Top_K                           
5                 1.28  0.6137  
10                1.32  0.6904  
15                1.64  0.7000  
20                1.64  0.9074

In [32]:
# =============================================================================
# STEP 29: FEATURE-SELECTION STABILITY
# =============================================================================

from itertools import combinations

stability_rows = []

for method in ["mRMR", "DODA"]:

    for top_k in TOP_K_VALUES:

        subset = cv_results_df[
            (cv_results_df["Method"] == method) &
            (cv_results_df["Top_K"] == top_k)
        ]

        # One feature set per run; model does not affect selection.
        run_sets = {}

        for run_id in range(1, 26):
            row = subset[
                subset["Run"] == run_id
            ].iloc[0]

            run_sets[run_id] = set(
                row["Selected_Features"].split(", ")
            )

        pairwise_jaccards = []

        for a, b in combinations(range(1, 26), 2):

            set_a = run_sets[a]
            set_b = run_sets[b]

            pairwise_jaccards.append(
                len(set_a & set_b) / len(set_a | set_b)
            )

        stability_rows.append({
            "Method": method,
            "Top_K": top_k,
            "Mean_Jaccard": np.mean(pairwise_jaccards),
            "SD_Jaccard": np.std(pairwise_jaccards, ddof=1),
            "Min_Jaccard": np.min(pairwise_jaccards),
            "Max_Jaccard": np.max(pairwise_jaccards)
        })

stability_df = pd.DataFrame(stability_rows)

display(stability_df.round(4))


,Method,Top_K,Mean_Jaccard,SD_Jaccard,Min_Jaccard,Max_Jaccard
0,mRMR,5,0.4062,0.1711,0.1111,1.0000
1,mRMR,10,0.4775,0.1215,0.2500,0.8182
2,mRMR,15,0.6147,0.0981,0.4286,1.0000
3,mRMR,20,0.7599,0.0622,0.6667,0.9048
4,DODA,5,0.3482,0.1589,0.1111,1.0000
5,DODA,10,0.5214,0.1222,0.2500,0.8182
6,DODA,15,0.6663,0.0976,0.4286,0.8750
7,DODA,20,0.8227,0.0687,0.6667,1.0000


## 4.1 Additional Feature-Selection Stability Analyses

The 25 repeated-CV runs are used to quantify selection frequency, chance-adjusted stability, and direct mRMR–DODA feature-set agreement.

In [33]:
# =============================================================================
# STEP 30: FEATURE SELECTION FREQUENCY
# =============================================================================
frequency_rows = []
for method in ["mRMR", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        counts = {feature: 0 for feature in X.columns}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            for feature in row["Selected_Features"].split(", "):
                counts[feature] += 1
        for feature, count in counts.items():
            frequency_rows.append({
                "Method": method, "Top_K": top_k, "Feature": feature,
                "Selection_Count": count,
                "Selection_Frequency": count / (N_SPLITS * N_REPEATS)
            })
feature_frequency_df = pd.DataFrame(frequency_rows)

display(
    feature_frequency_df.sort_values(
        ["Method", "Top_K", "Selection_Frequency", "Feature"],
        ascending=[True, True, False, True]
    ).groupby(["Method", "Top_K"], group_keys=False).head(10).round(4)
)


,Method,Top_K,Feature,Selection_Count,Selection_Frequency
97,DODA,5,Bp,25,1.00
106,DODA,5,Bu,13,0.52
111,DODA,5,Pcv,13,0.52
109,DODA,5,Pot,12,0.48
107,DODA,5,Sc,12,0.48
...,...,...,...,...,...
76,mRMR,20,Su,25,1.00
95,mRMR,20,Ane,24,0.96
93,mRMR,20,Appet,24,0.96
94,mRMR,20,Pe,24,0.96


In [34]:
# =============================================================================
# STEP 31: KUNCHEVA INDEX
# =============================================================================
from itertools import combinations
N_FEATURES = X.shape[1]
kuncheva_rows = []

for method in ["mRMR", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        run_sets = {}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            run_sets[run_id] = set(row["Selected_Features"].split(", "))

        for run_a, run_b in combinations(sorted(run_sets), 2):
            intersection = len(run_sets[run_a] & run_sets[run_b])
            denominator = top_k * (N_FEATURES - top_k)
            value = np.nan if denominator == 0 else ((intersection * N_FEATURES) - (top_k ** 2)) / denominator
            kuncheva_rows.append({
                "Method": method, "Top_K": top_k, "Run_A": run_a, "Run_B": run_b,
                "Intersection": intersection, "Kuncheva_Index": value
            })

kuncheva_df = pd.DataFrame(kuncheva_rows)
kuncheva_summary_df = (
    kuncheva_df.groupby(["Method", "Top_K"])["Kuncheva_Index"]
    .agg(Mean_Kuncheva="mean", Std_Kuncheva="std", Min_Kuncheva="min", Max_Kuncheva="max")
    .reset_index()
)
display(kuncheva_summary_df.round(4))


,Method,Top_K,Mean_Kuncheva,Std_Kuncheva,Min_Kuncheva,Max_Kuncheva
0,DODA,5,0.3642,0.2162,-0.0105,1.0000
1,DODA,10,0.4463,0.1809,-0.0286,0.8286
2,DODA,15,0.4548,0.1914,-0.0667,0.8222
3,DODA,20,0.4070,0.2471,-0.2000,1.0000
4,mRMR,5,0.4417,0.2086,-0.0105,1.0000
5,mRMR,10,0.3783,0.1897,-0.0286,0.8286
6,mRMR,15,0.3517,0.2000,-0.0667,1.0000
7,mRMR,20,0.1730,0.2399,-0.2000,0.7000


In [35]:
# =============================================================================
# STEP 32: mRMR–DODA FEATURE-SET AGREEMENT SUMMARY
# =============================================================================
agreement_summary_df = (
    feature_comparison_df.groupby("Top_K")
    .agg(
        Same_Feature_Set_Rate=("Same_Feature_Set", "mean"),
        Mean_Overlap=("Overlap_Count", "mean"),
        SD_Overlap=("Overlap_Count", "std"),
        Mean_Jaccard=("Jaccard", "mean"),
        SD_Jaccard=("Jaccard", "std"),
        Mean_Features_Changed=("Features_Changed", "mean"),
        SD_Features_Changed=("Features_Changed", "std")
    ).reset_index()
)
display(agreement_summary_df.round(4))


,Top_K,Same_Feature_Set_Rate,Mean_Overlap,SD_Overlap,Mean_Jaccard,SD_Jaccard,Mean_Features_Changed,SD_Features_Changed
0,5,0.04,3.72,0.6137,0.6062,0.1491,1.28,0.6137
1,10,0.08,8.68,0.6904,0.7731,0.1075,1.32,0.6904
2,15,0.04,13.36,0.7000,0.8060,0.0768,1.64,0.7000
3,20,0.04,18.36,0.9074,0.8515,0.0764,1.64,0.9074


In [36]:
# =============================================================================
# STEP 33: PAIRED STATISTICAL COMPARISON OF PREDICTIVE PERFORMANCE
# =============================================================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

comparison_rows = []

for top_k in TOP_K_VALUES:

    for model_name in models.keys():

        mrmr = cv_results_df[
            (cv_results_df["Method"] == "mRMR") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        doda = cv_results_df[
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        assert len(mrmr) == 25
        assert len(doda) == 25
        assert (mrmr["Run"].values == doda["Run"].values).all()

        for metric in metrics:

            mrmr_values = mrmr[metric].to_numpy()
            doda_values = doda[metric].to_numpy()

            differences = doda_values - mrmr_values

            mean_difference = np.mean(differences)
            sd_difference = np.std(differences, ddof=1)

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    doda_values,
                    mrmr_values,
                    alternative="two-sided"
                )

            paired_d = (
                mean_difference / sd_difference
                if sd_difference > 0 else 0.0
            )

            comparison_rows.append({
                "Top_K": top_k,
                "Model": model_name,
                "Metric": metric,
                "mRMR_Mean": np.mean(mrmr_values),
                "DODA_Mean": np.mean(doda_values),
                "Mean_Difference_DODA_minus_mRMR": mean_difference,
                "Paired_SD": sd_difference,
                "Paired_Cohens_d": paired_d,
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value
            })

performance_comparison_df = pd.DataFrame(comparison_rows)

reject, adjusted_p, _, _ = multipletests(
    performance_comparison_df["P_Value"],
    alpha=0.05,
    method="holm"
)

performance_comparison_df["Holm_Adjusted_P"] = adjusted_p
performance_comparison_df["Significant"] = reject

display(performance_comparison_df.round(4))


,Top_K,Model,Metric,mRMR_Mean,DODA_Mean,Mean_Difference_DODA_minus_mRMR,Paired_SD,Paired_Cohens_d,Wilcoxon_Statistic,P_Value,Holm_Adjusted_P,Significant
0,5,Logistic Regression,Accuracy,0.5170,0.5129,-0.0041,0.0220,-0.1847,91.5,0.4041,1.0000,False
1,5,Logistic Regression,Precision,0.5261,0.5228,-0.0032,0.0195,-0.1656,115.0,0.4842,1.0000,False
2,5,Logistic Regression,Recall,0.5799,0.5768,-0.0031,0.0325,-0.0961,121.0,0.8580,1.0000,False
3,5,Logistic Regression,F1,0.5512,0.5478,-0.0033,0.0232,-0.1435,133.0,0.8791,1.0000,False
4,5,Logistic Regression,ROC_AUC,0.5203,0.5156,-0.0047,0.0164,-0.2844,95.0,0.1909,1.0000,False
5,5,Random Forest,Accuracy,0.5020,0.5091,0.0071,0.0436,0.1627,113.5,0.2970,1.0000,False
6,5,Random Forest,Precision,0.5144,0.5207,0.0063,0.0406,0.1546,112.0,0.2776,1.0000,False
7,5,Random Forest,Recall,0.5242,0.5333,0.0091,0.0637,0.1431,114.5,0.4745,1.0000,False
8,5,Random Forest,F1,0.5187,0.5260,0.0074,0.0466,0.1579,114.0,0.3037,1.0000,False
9,5,Random Forest,ROC_AUC,0.5045,0.4965,-0.0080,0.0369,-0.2168,140.0,0.5602,1.0000,False


In [37]:
# =============================================================================
# STEP 34: SAVE STATISTICAL RESULTS
# =============================================================================

stats_dir = Path("../../../results/kidney_disease/statistics")
stats_dir.mkdir(parents=True, exist_ok=True)

feature_comparison_df.to_csv(
    stats_dir / "feature_set_comparison_5x5.csv",
    index=False
)

stability_df.to_csv(
    stats_dir / "feature_selection_stability_5x5.csv",
    index=False
)

performance_comparison_df.to_csv(
    stats_dir / "mrmr_vs_doda_predictive_statistics_5x5.csv",
    index=False
)

print(f"Saved statistical outputs to: {stats_dir.resolve()}")

feature_frequency_df.to_csv(
    stats_dir / "feature_selection_frequency_5x5.csv",
    index=False
)

kuncheva_df.to_csv(
    stats_dir / "kuncheva_pairwise_5x5.csv",
    index=False
)

kuncheva_summary_df.to_csv(
    stats_dir / "kuncheva_summary_5x5.csv",
    index=False
)

agreement_summary_df.to_csv(
    stats_dir / "mrmr_doda_agreement_summary_5x5.csv",
    index=False
)


Saved statistical outputs to: C:\Users\johnm\msc_research\results\kidney_disease\statistics


In [38]:
# =============================================================================
# STEP 36: FINAL SANITY CHECKS
# =============================================================================

print("=" * 70)
print("FINAL EXPERIMENT SANITY CHECK")
print("=" * 70)

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"5-fold CV result rows: {len(cv5_results_df)}")
print(f"Repeated CV result rows: {len(cv_results_df)}")
print(f"Expected 5-fold rows: {5 * len(TOP_K_VALUES) * 2 * len(models)}")
print(f"Expected repeated CV rows: {25 * len(TOP_K_VALUES) * 2 * len(models)}")

assert X.shape[1] == 24
assert X.isna().sum().sum() == 0
assert TOP_K_VALUES == [5, 10, 15, 20]
assert len(cv5_results_df) == 5 * len(TOP_K_VALUES) * 2 * len(models)
assert len(cv_results_df) == 25 * len(TOP_K_VALUES) * 2 * len(models)
assert len(feature_frequency_df) == 2 * len(TOP_K_VALUES) * X.shape[1]
assert len(kuncheva_summary_df) == 2 * len(TOP_K_VALUES)
assert len(agreement_summary_df) == len(TOP_K_VALUES)

for top_k in TOP_K_VALUES:
    assert len(mrmr_results[top_k]["features"]) == top_k
    assert len(rank_doda_results[top_k]["features"]) == top_k

print("All sanity checks passed.")


FINAL EXPERIMENT SANITY CHECK
Total predictors: 24
Top-K values: [5, 10, 15, 20]
Missing values: 0
5-fold CV result rows: 120
Repeated CV result rows: 600
Expected 5-fold rows: 120
Expected repeated CV rows: 600
All sanity checks passed.
